# Score a pair. Understand the result.

A gentle first run with the DecoderTCR **sequence model**.

You will identify the eight inputs, score a public example, inspect rows the model cannot reconstruct, and save results with their source and model information. A score helps rank sequence pairs; it does not tell you a percentage chance of binding.

Read the explanations even if a colleague runs the code for you. They describe what the output can support and what to ask when a row fails.

## Before you begin

This is an **adapted tutorial with input cells only**. It contains no measured model outputs and has not been executed. The API and sample inputs were checked against [DecoderTCR at `3e3d988`](https://github.com/czbiohub-chi/DecoderTCR/blob/3e3d9889d26d79635f674940ddab039c4dc6f9f9/README.md), sequence model V0.3. Running it will produce your own results, which depend on the selected checkpoint and computing environment.

You need Git, [uv](https://docs.astral.sh/uv/getting-started/installation/), an internet connection for setup, and enough memory and disk space for the selected checkpoint. CPU inference is supported; no completion time is promised. These instructions select the CPU example checkpoint explicitly. The project's usual default is `DecoderTCR-ESMC_600M`.

In a terminal, create a fresh source checkout and install its environment:

```bash
git clone https://github.com/czbiohub-chi/DecoderTCR.git
cd DecoderTCR
git checkout 3e3d9889d26d79635f674940ddab039c4dc6f9f9
uv sync
uv run stitchrdl -s human
uv run python scripts/download_weights.py -m DecoderTCR-ESMC_300M
uv run --with jupyter jupyter lab
```

Copy this notebook into the new `DecoderTCR` folder, open it in Jupyter, and run the cells in order. The `Python 3` kernel should use the environment launched above. See the [official uv/Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/). Keep `-m` in the download command: leaving it out downloads all registered checkpoints.

If you already have the repository and weights, use that checkout after checking its revision; do not overwrite your own changes. To use CUDA and the 600M checkpoint instead, download that checkpoint explicitly and change `MODEL` and `DEVICE` together below.

In [ ]:
from pathlib import Path
import json
import subprocess
import platform
from datetime import datetime, timezone

import DecoderTCR as dt
import pandas as pd
import torch

MODEL = "DecoderTCR-ESMC_300M"
DEVICE = "cpu"
SOURCE_REVISION = "3e3d9889d26d79635f674940ddab039c4dc6f9f9"
SOURCE_ROOT = Path(dt.__file__).resolve().parents[2]
actual_revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=SOURCE_ROOT, text=True
).strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError("This tutorial targets a different source revision. Check the API before continuing.")
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Set DEVICE to cpu or select a CUDA host.")

run_context = {
    "source_commit": actual_revision,
    "package_version": dt.__version__,
    "model": MODEL,
    "device": DEVICE,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "started_utc": datetime.now(timezone.utc).isoformat(),
}
print(run_context)

## The eight inputs

The starting record describes both parts of a receptor, the HLA allele, and a peptide. Gene names are text identifiers; CDR3 and peptide fields are amino-acid letters.

| Field | What you supply |
|---|---|
| `trav` | The alpha-chain V gene name |
| `traj` | The alpha-chain J gene name |
| `cdr3a` | The alpha-chain CDR3 amino-acid sequence |
| `trbv` | The beta-chain V gene name |
| `trbj` | The beta-chain J gene name |
| `cdr3b` | The beta-chain CDR3 amino-acid sequence |
| `hla` | The HLA allele identifier, such as `HLA-B*27:05` |
| `peptide` | The peptide amino-acid sequence to score |

The components API reconstructs the longer receptor chains and looks up the HLA sequence. You do not need to paste those full sequences yourself. An HLA allele name is not an amino-acid sequence: use this components API when supplying allele names. [Input contract and reconstruction](https://github.com/czbiohub-chi/DecoderTCR/blob/3e3d9889d26d79635f674940ddab039c4dc6f9f9/src/DecoderTCR/reconstruct/components.py).

In [ ]:
# Public example from the pinned repository README; no binding label is asserted here.
pair = {
    "trav": "TRAV21", "traj": "TRAJ6", "cdr3a": "CAVRPGGAGPFFVVF",
    "trbv": "TRBV7-9", "trbj": "TRBJ2-7", "cdr3b": "CASSLGQAYEQYF",
    "hla": "HLA-B*27:05", "peptide": "LRVMMLAPF",
}
pd.DataFrame([pair])

## Check the record before inference

The checks below catch empty fields and misspelled amino-acid characters. They do not prove that a receptor can be reconstructed or that the pair binds. The HLA reference only covers the alleles bundled with this source release.

In [ ]:
required = ("trav", "traj", "cdr3a", "trbv", "trbj", "cdr3b", "hla", "peptide")
missing = [key for key in required if not str(pair.get(key, "")).strip()]
if missing:
    raise ValueError(f"Missing fields: {missing}")

standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
for key in ("cdr3a", "cdr3b", "peptide"):
    unexpected = set(pair[key]) - standard_amino_acids
    if unexpected:
        raise ValueError(f"{key} contains unexpected characters: {sorted(unexpected)}")

available_alleles = dt.list_alleles()
print("Allele present in the reference:", pair["hla"] in available_alleles)

## Score the public example

This is the first cell that loads model weights and runs inference. The result is a table, even for a single pair. Keep its diagnostic columns.

In [ ]:
result = dt.score_from_components(pair, model=MODEL, device=DEVICE)
SCORE_COLUMN = f"pll_{MODEL}"
result[["name", "peptide", "ok", "tcr_reason", "hla_reason", SCORE_COLUMN]]

## Read the missing data before the scores

`ok=False` means the full complex could not be reconstructed. That row's NaN score is missing information; it is not a low score and does not mean “does not bind.” Read `tcr_reason` and `hla_reason`. Keep failed rows when saving results.

An unknown allele or an unstitchable receptor can cause a row to fail. An entire missing required column raises an error before scoring. [Failure handling](https://github.com/czbiohub-chi/DecoderTCR/blob/3e3d9889d26d79635f674940ddab039c4dc6f9f9/src/DecoderTCR/reconstruct/components.py).

In [ ]:
def describe_status(frame, score_column):
    total = len(frame)
    reconstructed = int(frame["ok"].sum())
    missing_scores = int(frame[score_column].isna().sum())
    print(f"{reconstructed}/{total} rows reconstructed; {missing_scores} scores missing.")
    for reason_column in ("tcr_reason", "hla_reason"):
        reasons = frame.loc[frame[reason_column].fillna("").ne(""), reason_column]
        if not reasons.empty:
            print(reason_column)
            print(reasons.value_counts().to_string())
    return frame.loc[~frame["ok"], ["name", "tcr_reason", "hla_reason"]]

describe_status(result, SCORE_COLUMN)

## What PLL means

DecoderTCR hides the whole peptide and reads the model's probabilities for its true amino acids. PLL is the average of their log-probabilities. A higher score means the model assigns the peptide higher likelihood in the supplied receptor and HLA context. It is not a probability of binding. [Scoring definition](https://github.com/czbiohub-chi/DecoderTCR/blob/3e3d9889d26d79635f674940ddab039c4dc6f9f9/src/DecoderTCR/utils/scoring.py).

This is **not a binding probability**. Exponentiating it does not make it a percentage chance of binding. It is not a measured affinity or an experimental confirmation. Keep the model, source revision, device and peptide length with your result. Do not mix scores from different checkpoints or treat different peptide contexts as calibrated to the same binding threshold.

The questions to ask are: Which checkpoint produced the score? Was the input reconstructed successfully? What comparison set is being ranked? What experiment would test the prediction?

## Try a small table

The repository's [sample CSV](https://github.com/czbiohub-chi/DecoderTCR/blob/3e3d9889d26d79635f674940ddab039c4dc6f9f9/Demo/sample_data/genes_pairs.csv) uses the same eight fields, plus `name` and `label`. The code below takes only its first three rows to demonstrate batch input; this subset is not a benchmark. Labels are supplied data, not model predictions.

For your own file, replace `sample_path`. Keep an untouched copy of the original data. If you supply a `name` column, each row should have a unique name.

In [ ]:
sample_path = SOURCE_ROOT / "Demo/sample_data/genes_pairs.csv"
pairs = pd.read_csv(sample_path, keep_default_na=False).head(3).copy()
if pairs["name"].duplicated().any():
    raise ValueError("Use unique row names before scoring.")
scored_pairs = dt.score_from_components(pairs, model=MODEL, device=DEVICE)
describe_status(scored_pairs, SCORE_COLUMN)
scored_pairs[["name", "hla", "peptide", "label", "ok", SCORE_COLUMN]]

## Save a result that someone else can inspect

The CSV retains all rows, including failures. The adjacent JSON file records how this run was configured. This example is for learning the software; published evaluations are listed on the benchmarks page.

In [ ]:
output_dir = SOURCE_ROOT / "tutorial-results" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir.mkdir(parents=True, exist_ok=False)
scored_pairs.to_csv(output_dir / "scored-pairs.csv", index=False)
run_record = {
    **run_context,
    "input": str(sample_path),
    "input_selection": "first three rows",
    "rows": len(scored_pairs),
    "reconstructed_rows": int(scored_pairs["ok"].sum()),
    "missing_scores": int(scored_pairs[SCORE_COLUMN].isna().sum()),
    "peptide_lengths": sorted(scored_pairs["peptide"].str.len().unique().tolist()),
    "mask_region": "peptide",
}
(output_dir / "run.json").write_text(json.dumps(run_record, indent=2) + "\n")
print(output_dir)

## When something goes wrong

| What you see | What to check next |
|---|---|
| `ModuleNotFoundError: DecoderTCR` | Launch Jupyter from the source checkout with its uv environment. |
| A missing checkpoint error | Run the download command with exactly the `MODEL` selected above. |
| A Stitchr germline-data error | Run `uv run stitchrdl -s human` in the source environment. |
| A non-empty `hla_reason` | Inspect `dt.list_alleles()` and your allele spelling. |
| A non-empty `tcr_reason` | Inspect the V/J genes and CDR3 sequence together. |
| CUDA is unavailable | Select `DEVICE = "cpu"` or use a CUDA host. |
| Scores differ from a published notebook | Check its checkpoint, source, device and precision before comparing. |

Continue with [Generate peptides for a specific TCR](https://tcrchallenge.org/tutorials/iegr-pssm/) to generate candidates with IEGR and summarize their position-specific preferences. The authors' [original notebook](https://github.com/czbiohub-chi/DecoderTCR/blob/3e3d9889d26d79635f674940ddab039c4dc6f9f9/Demo/quick_start.ipynb) provides the original walkthrough and its saved outputs. Source code and adapted examples are covered by the [repository MIT license](https://github.com/czbiohub-chi/DecoderTCR/blob/3e3d9889d26d79635f674940ddab039c4dc6f9f9/LICENSE.md).